# access smoke test

Spatial-accessibility metrics (PySAL `access`): floating-catchment-area family
(2SFCA / E2SFCA / 3SFCA / RAAM) exercised on a small synthetic
demand/supply/cost triple, so it runs offline (the built-in `Datasets.load_data`
Chicago files are multi-MB downloads).

In [ ]:
# Example 1: Construct an Access object from synthetic demand/supply/cost data.
# 12 synthetic "tracts" on a line; travel cost grows with tract distance.
import numpy as np, pandas as pd
from access import Access

rng = np.random.default_rng(0)
n = 12
demand = pd.DataFrame({'geoid': range(n), 'pop': rng.integers(100, 1000, n)})
supply = pd.DataFrame({'geoid': range(n), 'doc': rng.integers(1, 10, n),
                       'dentist': rng.integers(1, 10, n)})
cost = pd.DataFrame([(o, d, abs(o - d) * 5 + 1) for o in range(n) for d in range(n)],
                    columns=['origin', 'dest', 'cost'])
acc = Access(demand_df=demand, demand_index='geoid', demand_value='pop',
             supply_df=supply, supply_index='geoid', supply_value=['doc', 'dentist'],
             cost_df=cost, cost_origin='origin', cost_dest='dest', cost_name='cost')

print(f"demand: {len(acc.demand_df)} tracts | supply: {len(acc.supply_df)} sites "
      f"| cost pairs: {len(acc.cost_df)}")
assert len(acc.demand_df) == n and len(acc.supply_df) == n
assert len(acc.cost_df) == n * n
assert set(acc.cost_df.columns) == {'origin', 'dest', 'cost'}
print("Access object constructed from synthetic offline data")


In [ ]:
# Example 2: Two-Step Floating Catchment Area (2SFCA) — the headline method.
import numpy as np, pandas as pd
from access import Access

rng = np.random.default_rng(0)
n = 12
demand = pd.DataFrame({'geoid': range(n), 'pop': rng.integers(100, 1000, n)})
supply = pd.DataFrame({'geoid': range(n), 'doc': rng.integers(1, 10, n),
                       'dentist': rng.integers(1, 10, n)})
cost = pd.DataFrame([(o, d, abs(o - d) * 5 + 1) for o in range(n) for d in range(n)],
                    columns=['origin', 'dest', 'cost'])
acc = Access(demand_df=demand, demand_index='geoid', demand_value='pop',
             supply_df=supply, supply_index='geoid', supply_value=['doc', 'dentist'],
             cost_df=cost, cost_origin='origin', cost_dest='dest', cost_name='cost')

acc.two_stage_fca(name='2sfca', max_cost=30.0)
cols = list(acc.access_df.columns)
print(f"access_df columns: {cols}")
assert '2sfca_doc' in cols and '2sfca_dentist' in cols
assert (acc.access_df['2sfca_doc'].dropna() >= 0).all()
print(acc.access_df.head(3))
print("2SFCA computed for both supply columns")


In [ ]:
# Example 3: Enhanced 2SFCA — gaussian distance-decay weights.
from access import weights
import numpy as np, pandas as pd
from access import Access

rng = np.random.default_rng(0)
n = 12
demand = pd.DataFrame({'geoid': range(n), 'pop': rng.integers(100, 1000, n)})
supply = pd.DataFrame({'geoid': range(n), 'doc': rng.integers(1, 10, n),
                       'dentist': rng.integers(1, 10, n)})
cost = pd.DataFrame([(o, d, abs(o - d) * 5 + 1) for o in range(n) for d in range(n)],
                    columns=['origin', 'dest', 'cost'])
acc = Access(demand_df=demand, demand_index='geoid', demand_value='pop',
             supply_df=supply, supply_index='geoid', supply_value=['doc', 'dentist'],
             cost_df=cost, cost_origin='origin', cost_dest='dest', cost_name='cost')

acc.enhanced_two_stage_fca(name='e2sfca', weight_fn=weights.gaussian(15), max_cost=30.0)
cols = list(acc.access_df.columns)
print(f"access_df columns: {cols}")
assert 'e2sfca_doc' in cols and 'e2sfca_dentist' in cols
assert (acc.access_df['e2sfca_doc'].dropna() >= 0).all()
print("E2SFCA with gaussian decay computed")


In [ ]:
# Example 4: Three-Step FCA (provider-selection weights) + the RAAM model.
import numpy as np, pandas as pd
from access import Access

rng = np.random.default_rng(0)
n = 12
demand = pd.DataFrame({'geoid': range(n), 'pop': rng.integers(100, 1000, n)})
supply = pd.DataFrame({'geoid': range(n), 'doc': rng.integers(1, 10, n),
                       'dentist': rng.integers(1, 10, n)})
cost = pd.DataFrame([(o, d, abs(o - d) * 5 + 1) for o in range(n) for d in range(n)],
                    columns=['origin', 'dest', 'cost'])
acc = Access(demand_df=demand, demand_index='geoid', demand_value='pop',
             supply_df=supply, supply_index='geoid', supply_value=['doc', 'dentist'],
             cost_df=cost, cost_origin='origin', cost_dest='dest', cost_name='cost')

acc.three_stage_fca(name='3sfca', max_cost=30.0)
acc.raam(name='raam', tau=30)
cols = set(acc.access_df.columns)
print(f"access_df columns: {sorted(cols)}")
assert {'3sfca_doc', '3sfca_dentist', 'raam_doc', 'raam_dentist'} <= cols
assert (acc.access_df['3sfca_doc'].dropna() >= 0).all()
print("3SFCA + RAAM computed")


In [ ]:
# Example 5: Multiple thresholds + a normalized composite score across measures.
import numpy as np, pandas as pd
from access import Access

rng = np.random.default_rng(0)
n = 12
demand = pd.DataFrame({'geoid': range(n), 'pop': rng.integers(100, 1000, n)})
supply = pd.DataFrame({'geoid': range(n), 'doc': rng.integers(1, 10, n),
                       'dentist': rng.integers(1, 10, n)})
cost = pd.DataFrame([(o, d, abs(o - d) * 5 + 1) for o in range(n) for d in range(n)],
                    columns=['origin', 'dest', 'cost'])
acc = Access(demand_df=demand, demand_index='geoid', demand_value='pop',
             supply_df=supply, supply_index='geoid', supply_value=['doc', 'dentist'],
             cost_df=cost, cost_origin='origin', cost_dest='dest', cost_name='cost')

acc.two_stage_fca(name='a20', max_cost=20.0)
acc.two_stage_fca(name='a40', max_cost=40.0)
score = acc.score(col_dict={'a20_doc': 1, 'a40_doc': 1})
print(f"norm_access_df columns: {list(acc.norm_access_df.columns)}")
print(f"composite score head: {[round(v, 3) for v in score.head(3)]}")
assert len(score) == n and score.notna().all()
assert {'a20_doc', 'a40_doc'} <= set(acc.access_df.columns)
print("multi-threshold comparison + composite score computed")
